# FastAPI for Model Serving

**FastAPI** is a modern, high-performance Python web framework for building HTTP/JSON
APIs, built on **Starlette** (ASGI) and **Pydantic** (data validation). In the
MLOps world it is the default way to wrap a trained model in a REST endpoint:
you load the model once, expose a `/predict` route, validate requests with a typed
schema, and serve it with an ASGI server such as **uvicorn** or **gunicorn**.

It powers the API layer of many inference stacks — Hugging Face Inference Endpoints,
Ray Serve, BentoML, and LitServe all speak FastAPI/Starlette under the hood — so
understanding it is the foundation for nearly every Python model-serving pattern.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

<a id='introduction'></a>

### What is it?

FastAPI is an ASGI web framework for building APIs with standard Python type hints.
You declare your request and response shapes as Pydantic models / type annotations,
and FastAPI gives you for free: request parsing, validation, serialization,
automatic OpenAPI (Swagger) docs, and first-class `async` support. For ML serving
it is the thin, fast, well-documented layer between an HTTP client and your model's
`predict()` call.

### Why use it?

- **Speed (dev and runtime):** built on Starlette + Pydantic; among the fastest
  Python frameworks, and async lets one worker handle many concurrent I/O-bound
  requests (e.g. calls out to a GPU service or vector DB).
- **Type-driven validation:** a bad request (missing field, wrong dtype) is rejected
  with a clear 422 before it ever reaches your model.
- **Auto docs:** interactive Swagger UI at `/docs` and ReDoc at `/redoc`, generated
  from your types — invaluable for handing an endpoint to another team.
- **Ecosystem fit:** background tasks, dependency injection, WebSockets, streaming
  responses (for token-by-token LLM output), and easy middleware.

### When to use it (and when not)?

Reach for FastAPI when you need a **custom inference API**: bespoke pre/post-processing,
multiple models behind one service, business logic around the prediction, or streaming.

Reach for something else when a purpose-built server already does the job: **vLLM**,
**TGI**, or **Triton** for high-throughput LLM/GPU serving with continuous batching;
**TorchServe**/**TensorFlow Serving** for framework-native model management; **BentoML**
or **LitServe** when you want batching, packaging, and autoscaling out of the box
(both build on top of FastAPI/Starlette anyway).

## Key Features

<a id='key-features'></a>

| Feature | Description | Why it matters for serving |
|---------|-------------|----------------------------|
| Pydantic schemas | Request/response models with typed fields and validators | Reject malformed input with a 422 before it hits the model |
| Async / await | Native ASGI coroutine support | One worker overlaps many I/O-bound calls (DBs, downstream models) |
| Dependency injection | `Depends()` for shared resources (model, auth, DB session) | Clean way to inject a loaded model or API-key check |
| Automatic OpenAPI | Swagger UI `/docs`, ReDoc `/redoc`, JSON schema `/openapi.json` | Self-documenting, testable endpoints with zero extra code |
| Lifespan events | `lifespan` context loads the model once at startup | Avoid reloading weights on every request |
| Streaming responses | `StreamingResponse` / SSE | Token-by-token LLM output and large payloads |
| Background tasks | Run work after returning the response | Logging, metrics, cache writes without blocking the client |
| Middleware & CORS | ASGI middleware stack | Auth, GZip, request-ID, CORS for browser clients |

## Architecture Overview

<a id='architecture'></a>

A FastAPI inference service is a layered stack. Requests flow top-to-bottom and
responses flow back up:

```
   HTTP client  (curl / browser / another service)
        |  JSON over HTTP
        v
   +---------------------------------------------+
   |  Process manager: gunicorn / uvicorn workers|  <- N OS processes
   +---------------------------------------------+
        v
   +---------------------------------------------+
   |  ASGI server (uvicorn)  - event loop        |
   +---------------------------------------------+
        v
   +---------------------------------------------+
   |  Starlette: routing, middleware, lifespan   |
   +---------------------------------------------+
        v
   +---------------------------------------------+
   |  FastAPI: Pydantic validate -> your handler |
   |           -> model.predict() -> serialize   |
   +---------------------------------------------+
```

### Components

1. **ASGI app (`FastAPI()`):** the application object holding routes, middleware,
   and the lifespan handler.
2. **ASGI server (uvicorn):** runs the event loop, translates raw HTTP into ASGI
   `scope`/`receive`/`send` events. In production it's typically supervised by
   **gunicorn** with the `uvicorn.workers.UvicornWorker` class to run multiple
   worker processes.
3. **Pydantic models:** define and validate the wire format; also drive the OpenAPI
   schema and response serialization.
4. **The model object:** loaded **once** in the lifespan handler and stored in
   `app.state` (or a module global), then reused by every request — never reloaded
   per call.

## Installation

<a id='installation'></a>

### Prerequisites

- Python 3.8+ (3.10+ recommended).
- An ASGI server: **uvicorn** for development/single-node, **gunicorn + uvicorn
  workers** for production.
- Your model's runtime (PyTorch / scikit-learn / ONNX Runtime / etc.). GPU serving
  additionally needs the matching CUDA build of your framework.

`fastapi[standard]` pulls in uvicorn, the `fastapi` CLI, and useful extras
(`httpx`, `jinja2`, `python-multipart`). The examples below also use
scikit-learn for a tiny CPU model and `TestClient` (which needs `httpx`).

In [ ]:
# Install FastAPI + an ASGI server + an HTTP client for testing.
# In Colab/Jupyter the leading % makes pip target THIS kernel.
%pip install -q "fastapi[standard]" "uvicorn[standard]" httpx scikit-learn

import fastapi, starlette, pydantic
print("fastapi   ", fastapi.__version__)
print("starlette ", starlette.__version__)
print("pydantic  ", pydantic.__version__)

## Basic Usage

<a id='basic-usage'></a>

### Quick start: train a tiny model, wrap it in an endpoint, call it

We'll train a 4-feature Iris classifier, define typed request/response schemas, and
expose a `/predict` route. Instead of starting a real server (awkward inside a
notebook), we drive the app **in-process** with `TestClient`, which sends real ASGI
requests through the full FastAPI stack — validation, routing, serialization — with
no network involved.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

# --- train a small CPU model once (stand-in for 'load my trained weights') ---
iris = load_iris()
clf = LogisticRegression(max_iter=1000).fit(iris.data, iris.target)
CLASS_NAMES = list(iris.target_names)

# --- typed wire format: validation + docs come for free ---
class IrisRequest(BaseModel):
    sepal_length: float = Field(..., ge=0, le=10, examples=[5.1])
    sepal_width:  float = Field(..., ge=0, le=10, examples=[3.5])
    petal_length: float = Field(..., ge=0, le=10, examples=[1.4])
    petal_width:  float = Field(..., ge=0, le=10, examples=[0.2])

class IrisResponse(BaseModel):
    label: str
    class_id: int
    probabilities: dict[str, float]

app = FastAPI(title="Iris Classifier", version="1.0.0")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=IrisResponse)
def predict(req: IrisRequest):
    x = [[req.sepal_length, req.sepal_width, req.petal_length, req.petal_width]]
    proba = clf.predict_proba(x)[0]
    class_id = int(proba.argmax())
    return IrisResponse(
        label=CLASS_NAMES[class_id],
        class_id=class_id,
        probabilities={n: round(float(p), 4) for n, p in zip(CLASS_NAMES, proba)},
    )

print("app ready:", [r.path for r in app.routes if getattr(r, 'methods', None)])

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# happy path
r = client.post("/predict", json={
    "sepal_length": 5.1, "sepal_width": 3.5,
    "petal_length": 1.4, "petal_width": 0.2,
})
print("200 ->", r.json())

# health check
print("health ->", client.get("/health").json())

# validation failure: missing field -> 422 (model never runs)
bad = client.post("/predict", json={"sepal_length": 5.1})
print("bad request ->", bad.status_code, bad.json()["detail"][0]["msg"])

## Advanced Features

<a id='advanced-features'></a>

### 1. Load the model once with `lifespan`

Loading weights on import (module global) works but couples model loading to import
time. The idiomatic pattern is the **lifespan** context manager: load on startup,
stash on `app.state`, release on shutdown. Inject it into handlers with `Depends`
so tests can override it.

### 2. `async def` vs `def` handlers

- Use **`async def`** when the handler `await`s I/O (downstream HTTP, async DB).
- Use plain **`def`** for CPU-bound or blocking work (most `model.predict()` calls):
  FastAPI runs `def` handlers in a threadpool so they don't block the event loop.
- The cardinal sin is a blocking call (sync `predict`, `time.sleep`, sync `requests`)
  inside an `async def` — it stalls the whole event loop for every other request.

### 3. Streaming responses (LLM tokens)

`StreamingResponse` over a generator lets you flush tokens as they're produced,
the basis of the typewriter effect in chat UIs (often as Server-Sent Events).

In [ ]:
from contextlib import asynccontextmanager
from fastapi import Depends, Request
from fastapi.responses import StreamingResponse
import asyncio, json as _json

@asynccontextmanager
async def lifespan(app: FastAPI):
    # startup: load weights ONCE (here we reuse the clf trained above)
    app.state.model = clf
    print("[lifespan] model loaded")
    yield
    # shutdown: free resources (close sessions, release GPU, etc.)
    app.state.model = None
    print("[lifespan] model released")

app2 = FastAPI(lifespan=lifespan)

def get_model(request: Request):
    return request.app.state.model      # injected, easy to override in tests

@app2.post("/predict")
def predict2(req: IrisRequest, model=Depends(get_model)):
    x = [[req.sepal_length, req.sepal_width, req.petal_length, req.petal_width]]
    return {"class_id": int(model.predict(x)[0])}

# streaming: simulate token-by-token generation as Server-Sent Events
async def token_stream(prompt: str):
    for tok in (prompt + " -> generated answer").split():
        await asyncio.sleep(0)        # yield control to the event loop
        yield f"data: {_json.dumps({'token': tok})}\n\n"

@app2.get("/generate")
async def generate(prompt: str = "hello"):
    return StreamingResponse(token_stream(prompt), media_type="text/event-stream")

# 'with TestClient(...)' triggers the lifespan startup/shutdown events
with TestClient(app2) as c:
    print("predict ->", c.post("/predict", json={
        "sepal_length": 6.7, "sepal_width": 3.0,
        "petal_length": 5.2, "petal_width": 2.3}).json())
    print("stream  ->", c.get("/generate?prompt=hi").text.strip().splitlines()[:2])

## Use Cases

<a id='use-cases'></a>

#### Use Case 1: Real-time online inference

- **Context:** a product needs sub-second predictions (fraud score, recommendation,
  classification) behind a synchronous request.
- **Implementation:** load the model in `lifespan`; one `POST /predict` with a Pydantic
  schema; run `def` handlers in the threadpool; scale with gunicorn workers + a load
  balancer; add a `/health` route for the orchestrator's readiness probe.
- **Result:** a typed, documented, horizontally-scalable endpoint with input validation
  for free.

#### Use Case 2: LLM gateway with streaming

- **Context:** a chat backend that calls a downstream model server (vLLM/TGI) or an
  external API and needs to stream tokens to the browser.
- **Implementation:** `async def` handler that `await`s the upstream; relay tokens via
  `StreamingResponse`/SSE; use middleware for auth and request IDs.
- **Result:** low time-to-first-token, many concurrent streams per worker because the
  work is I/O-bound and async.

#### Use Case 3: Model-as-a-microservice in an MLOps platform

- **Context:** each model is its own service, containerized and deployed to Kubernetes,
  fronted by an API gateway.
- **Implementation:** FastAPI app in a Docker image, `/health` + `/metrics` (Prometheus)
  routes, structured logging, versioned schemas. Tools like BentoML/LitServe wrap this
  pattern and add batching/packaging.
- **Result:** independent deploy/scale per model, observable and self-documenting.

## Best Practices

<a id='best-practices'></a>

1. **Load the model once, in `lifespan`** — never inside the request handler. Reloading
   weights per request is the single most common serving mistake.
2. **Validate everything with Pydantic** — constrain ranges/shapes with `Field(...)` so
   garbage requests fail fast with a 422 instead of crashing the model.
3. **Pick `def` vs `async def` deliberately** — blocking model calls go in plain `def`
   (threadpool); only `await`-ing handlers should be `async def`. Never block the loop.
4. **Set an explicit `response_model`** — guarantees the output schema, strips extra
   fields, and documents the contract.
5. **Add `/health` (liveness/readiness) and `/metrics`** — orchestrators and dashboards
   depend on them.
6. **Version your API and schemas** (`/v1/predict`) so you can evolve without breaking
   clients; keep request/response models in a shared module.
7. **Test with `TestClient`** — fast, in-process, covers validation + handler logic in CI.
8. **Pin dependencies and the model artifact** — reproducible images; load weights from a
   versioned path/registry, not 'latest'.

## Common Pitfalls

<a id='pitfalls'></a>

1. **Blocking call inside `async def`** — a synchronous `model.predict()`, `requests.get`,
   or `time.sleep` in an `async` handler freezes the event loop and tanks concurrency for
   *every* client. Use `def` (threadpool) or `await run_in_threadpool(...)`.
2. **Reloading the model per request** — loads weights from disk on every call; latency
   explodes. Load once in `lifespan` / module scope.
3. **`--reload` or many workers in production without sizing** — `--reload` is a dev-only
   feature; and each gunicorn worker holds its own copy of the model, so `workers=8` with
   a 6 GB model = 48 GB RAM. Size workers to memory, not just CPU cores.
4. **Expecting in-process batching for free** — FastAPI handles one request per call; it
   does *not* batch across requests. For GPU throughput you need explicit micro-batching
   (a queue + batcher) or a server that does it (vLLM, Triton, LitServe, Ray Serve).
5. **Leaking internals in errors** — returning raw stack traces. Catch and return clean
   `HTTPException`s; log the detail server-side.
6. **Forgetting CORS** — browser clients get blocked until you add `CORSMiddleware`.

## Performance Optimization

<a id='performance'></a>

### Configuration tuning

- **Workers:** with gunicorn, `workers = (2 x cores) + 1` is the classic starting point
  for CPU/I/O-bound apps — but for ML, **memory per model copy** usually dominates, so
  pick the largest worker count your RAM/VRAM allows and load-test from there.
- **Threadpool size:** sync (`def`) handlers run in Starlette's `AnyIO` threadpool
  (default 40 threads). Raise it if you have many concurrent blocking predicts and CPU
  headroom; lower it to avoid oversubscription on a GIL-bound workload.
- **Micro-batching:** for GPU models, accumulate requests for a few milliseconds and run
  one batched forward pass — the biggest single throughput win. Implement with an async
  queue or adopt a server that batches (LitServe, Ray Serve, BentoML, vLLM).
- **Faster JSON / payloads:** use `ORJSONResponse` for big numeric payloads; add
  `GZipMiddleware` for large responses.
- **Move heavy work off-process:** offload non-critical work (logging, writes) to
  `BackgroundTasks`; keep the request path lean.

The cell below micro-benchmarks the in-process endpoint to show how to measure latency
and throughput (numbers are illustrative — a real benchmark uses a load tool like
`wrk`, `hey`, or `locust` against a running uvicorn server).

In [ ]:
import statistics, time

client = TestClient(app)
payload = {"sepal_length": 5.1, "sepal_width": 3.5,
           "petal_length": 1.4, "petal_width": 0.2}

# warm up
for _ in range(20):
    client.post("/predict", json=payload)

lat = []
N = 500
t0 = time.perf_counter()
for _ in range(N):
    s = time.perf_counter()
    client.post("/predict", json=payload)
    lat.append((time.perf_counter() - s) * 1000)  # ms
wall = time.perf_counter() - t0

lat.sort()
print(f"requests      : {N}")
print(f"throughput    : {N / wall:,.0f} req/s (in-process, single thread)")
print(f"latency  p50  : {statistics.median(lat):.3f} ms")
print(f"latency  p95  : {lat[int(0.95 * N)]:.3f} ms")
print(f"latency  p99  : {lat[int(0.99 * N)]:.3f} ms")

## Production Deployment

<a id='deployment'></a>

### Running the server

For a single node or container, run uvicorn directly. For multiple workers under a
process manager, run gunicorn with the uvicorn worker class:

```bash
# dev (single process, auto-reload)
uvicorn app:app --host 0.0.0.0 --port 8000 --reload

# production (4 worker processes, each an event loop)
gunicorn app:app -k uvicorn.workers.UvicornWorker \
    --workers 4 --bind 0.0.0.0:8000 --timeout 120
```

### Docker deployment

```dockerfile
FROM python:3.11-slim
WORKDIR /app

# install deps first for better layer caching
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# app code + model artifact
COPY . .

EXPOSE 8000
# exec form so signals (SIGTERM) reach gunicorn for graceful shutdown
CMD ["gunicorn", "app:app", "-k", "uvicorn.workers.UvicornWorker", \
     "--workers", "2", "--bind", "0.0.0.0:8000", "--timeout", "120"]
```

### Kubernetes deployment

Note the **readiness/liveness probes** pointing at `/health` and a **memory request
sized for one model copy per worker**.

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-classifier
spec:
  replicas: 3
  selector:
    matchLabels: { app: iris-classifier }
  template:
    metadata:
      labels: { app: iris-classifier }
    spec:
      containers:
        - name: api
          image: registry.example.com/iris-classifier:1.0.0
          ports:
            - containerPort: 8000
          resources:
            requests: { cpu: "500m", memory: "1Gi" }
            limits:   { cpu: "2",    memory: "2Gi" }
          readinessProbe:
            httpGet: { path: /health, port: 8000 }
            initialDelaySeconds: 10
            periodSeconds: 5
          livenessProbe:
            httpGet: { path: /health, port: 8000 }
            initialDelaySeconds: 20
            periodSeconds: 15
---
apiVersion: v1
kind: Service
metadata:
  name: iris-classifier
spec:
  selector: { app: iris-classifier }
  ports:
    - port: 80
      targetPort: 8000
```

## Monitoring and Observability

<a id='monitoring'></a>

#### Key metrics to track

- **Request rate (RPS)** — throughput per route and status code.
- **Latency p50 / p95 / p99** — tail latency matters more than the average for SLAs.
- **Error rate** — 4xx (client/validation) vs 5xx (server) split.
- **Saturation** — CPU/GPU utilization, memory, threadpool/queue depth, in-flight requests.
- **Model-specific** — prediction distribution, confidence, and input drift over time.

#### Instrumentation

- Use **`prometheus-fastapi-instrumentator`** to expose `/metrics` with latency and
  request counters in two lines, then scrape it with Prometheus + Grafana.
- Add **OpenTelemetry** (`opentelemetry-instrumentation-fastapi`) for distributed traces
  across the gateway -> model -> datastore path.

#### Logging best practices

- **Structure logs as JSON** (timestamp, level, route, status, latency_ms, request_id)
  so they're queryable in your log stack.
- **Attach a request ID** via middleware and propagate it downstream for correlation.
- **Use levels deliberately:** INFO for request summaries, WARNING for validation/4xx,
  ERROR (with stack trace, server-side only) for 5xx. Never log raw secrets or PII.

## Troubleshooting

<a id='troubleshooting'></a>

#### Issue 1: One slow request stalls all others

**Symptoms:** latency spikes and timeouts under load even though CPU is idle.

**Cause:** a blocking call (sync `predict`, `requests`, `time.sleep`) inside an
`async def` handler, which blocks the single event loop.

**Solution:** make the handler a plain `def` (FastAPI runs it in a threadpool), or
wrap the blocking call in `await starlette.concurrency.run_in_threadpool(...)`.

#### Issue 2: Out-of-memory / OOMKilled with multiple workers

**Symptoms:** container restarts; `workers=N` multiplies RAM/VRAM usage.

**Cause:** every gunicorn/uvicorn worker is a separate process loading its own copy
of the model weights.

**Solution:** size workers to `memory_limit / model_size`; for big GPU models use one
worker per GPU and scale with replicas, or move to a server with shared weights and
batching (Triton, Ray Serve).

#### Issue 3: 422 Unprocessable Entity on valid-looking requests

**Symptoms:** clients get 422 with a `detail` list.

**Cause:** the JSON body doesn't match the Pydantic schema (missing field, wrong type,
value outside a `Field` constraint).

**Solution:** read the `detail` array — it pinpoints the offending field and reason.
Check `/docs` for the exact expected schema; align client and server models.

## Comparison with Alternatives

<a id='comparison'></a>

| Aspect | FastAPI | Flask | BentoML / LitServe | vLLM / TGI / Triton |
|--------|---------|-------|--------------------|---------------------|
| Style | ASGI, async-first | WSGI, sync | Serving framework on FastAPI | Purpose-built inference server |
| Validation/docs | Built-in (Pydantic + OpenAPI) | Manual / extensions | Inherited from FastAPI | N/A (fixed API) |
| Request batching | DIY | DIY | Built-in adaptive batching | Built-in continuous batching |
| Best for | Custom inference APIs, gateways, streaming | Simple legacy sync apps | Packaging + batching + deploy | High-throughput LLM/GPU serving |
| Control vs convenience | Full control, some glue | Full control, more glue | Convenience, opinionated | Convenience, model-specific |

### When to choose FastAPI

- You need a **custom API** around the model: bespoke pre/post-processing, multiple
  models, business logic, auth, or streaming.
- You want **typed validation, auto docs, and async** without adopting a heavier
  framework.
- You're building a **gateway/orchestration layer** in front of dedicated inference
  servers.

Prefer a dedicated server (vLLM/TGI/Triton) when raw GPU throughput with continuous
batching is the goal, or a packaging framework (BentoML/LitServe) when you want
batching, autoscaling, and artifact management out of the box.

## Resources

<a id='resources'></a>

### Official documentation

- FastAPI docs: https://fastapi.tiangolo.com/
- FastAPI GitHub: https://github.com/fastapi/fastapi
- Starlette (ASGI toolkit underneath): https://www.starlette.io/
- Pydantic (validation): https://docs.pydantic.dev/
- Uvicorn (ASGI server): https://www.uvicorn.org/

### Tutorials and guides

- Official tutorial (start here): https://fastapi.tiangolo.com/tutorial/
- Deployment guide (servers, workers, Docker): https://fastapi.tiangolo.com/deployment/
- Lifespan events (load models on startup): https://fastapi.tiangolo.com/advanced/events/

### Serving ecosystem (build on / compare with FastAPI)

- BentoML: https://docs.bentoml.com/
- LitServe (Lightning): https://github.com/Lightning-AI/litserve
- Ray Serve: https://docs.ray.io/en/latest/serve/index.html
- prometheus-fastapi-instrumentator: https://github.com/trallnag/prometheus-fastapi-instrumentator

### Related notebooks

- `model-serving-libraries/deepspeed-mii.ipynb` — high-throughput LLM serving engine.
- Other `11-devops-mlops-infra/model-serving-libraries/` notebooks for Triton, vLLM, etc.